<a href="https://colab.research.google.com/github/VarshaP-0405/NLP-Skill-Hometask/blob/main/LSTM_Customer_Review_Sentiment_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torchtext scikit-learn pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 1.4 MB/s eta 0:00:00


In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
print("Dataset shape:", df.shape)
print(df.head())
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print("\nClass distribution:")
print(df["sentiment"].value_counts())
def clean_text(text):

    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<br\s*/?>", " ", text)

    # Keep only alphabets
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["review"] = df["review"].apply(clean_text)

print("\nCleaned example:")
print(df["review"].iloc[0][:300])
def tokenize(text):
    return text.split()


df["tokens"] = df["review"].apply(tokenize)

print("\nTokenized example:")
print(df["tokens"].iloc[0][:20])
train_reviews, test_reviews, train_labels, test_labels = train_test_split(
    df["tokens"].tolist(),
    df["sentiment"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)
train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    train_reviews,
    train_labels,
    test_size=0.1,
    random_state=42,
    stratify=train_labels
)

print("\nDataset split:")
print("Training:", len(train_reviews))
print("Validation:", len(val_reviews))
print("Testing:", len(test_reviews))
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
UNK_IDX = 1

word_counter = {}

for review in train_reviews:

    for word in review:

        word_counter[word] = word_counter.get(word, 0) + 1
MIN_FREQ = 2

vocab = {
    PAD_TOKEN: PAD_IDX,
    UNK_TOKEN: UNK_IDX
}

for word, count in word_counter.items():

    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

print("\nVocabulary size:", len(vocab))
def numericalize(tokens):

    return [
        vocab.get(word, UNK_IDX)
        for word in tokens
    ]
MAX_LEN = 200
def pad_sequence(tokens):

    sequence = numericalize(tokens)

    # Truncate long reviews
    sequence = sequence[:MAX_LEN]

    # Padding
    if len(sequence) < MAX_LEN:

        sequence += [
            PAD_IDX
        ] * (MAX_LEN - len(sequence))

    return sequence
class IMDBDataset(Dataset):

    def __init__(self, reviews, labels):

        self.reviews = reviews
        self.labels = labels

    def __len__(self):

        return len(self.reviews)

    def __getitem__(self, index):

        review = pad_sequence(
            self.reviews[index]
        )

        label = self.labels[index]

        return (
            torch.tensor(review, dtype=torch.long),
            torch.tensor(label, dtype=torch.float)
        )


train_dataset = IMDBDataset(
    train_reviews,
    train_labels
)

val_dataset = IMDBDataset(
    val_reviews,
    val_labels
)

test_dataset = IMDBDataset(
    test_reviews,
    test_labels
)
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)
class SentimentLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3
    ):

        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        # Fully connected layer
        self.fc = nn.Linear(
            hidden_dim,
            1
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Embedding
        embedded = self.embedding(x)

        # LSTM
        output, (hidden, cell) = self.lstm(
            embedded
        )

        # Last hidden state
        hidden = hidden[-1]

        hidden = self.dropout(hidden)

        # Fully connected layer
        output = self.fc(hidden)

        return output.squeeze(1)
model = SentimentLSTM(
    vocab_size=len(vocab)
).to(device)

print("\nModel:")
print(model)
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
def train_model(model, loader):

    model.train()

    total_loss = 0

    for reviews, labels in loader:

        reviews = reviews.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        predictions = model(reviews)

        loss = criterion(
            predictions,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)
def validate_model(model, loader):

    model.eval()

    total_loss = 0

    predictions_list = []
    labels_list = []

    with torch.no_grad():

        for reviews, labels in loader:

            reviews = reviews.to(device)
            labels = labels.to(device)

            predictions = model(reviews)

            loss = criterion(
                predictions,
                labels
            )

            total_loss += loss.item()

            probabilities = torch.sigmoid(
                predictions
            )

            predicted_classes = (
                probabilities >= 0.5
            ).int()

            predictions_list.extend(
                predicted_classes.cpu().numpy()
            )

            labels_list.extend(
                labels.cpu().numpy()
            )

    accuracy = accuracy_score(
        labels_list,
        predictions_list
    )

    return (
        total_loss / len(loader),
        accuracy
    )
EPOCHS = 5

train_losses = []
val_losses = []
val_accuracies = []

print("\nStarting training...\n")

for epoch in range(EPOCHS):

    train_loss = train_model(
        model,
        train_loader
    )

    val_loss, val_accuracy = validate_model(
        model,
        val_loader
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Training Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f} | "
        f"Validation Accuracy: {val_accuracy * 100:.2f}%"
    )
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for reviews, labels in test_loader:

        reviews = reviews.to(device)

        outputs = model(reviews)

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).int()

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions
)

recall = recall_score(
    all_labels,
    all_predictions
)

f1 = f1_score(
    all_labels,
    all_predictions
)


print("\n======================================")
print("       MODEL EVALUATION REPORT")
print("======================================")

print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")

print("======================================")
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    train_losses,
    marker="o",
    label="Training Loss"
)

plt.plot(
    range(1, EPOCHS + 1),
    val_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.grid()

plt.show()
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    val_accuracies,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")

plt.grid()

plt.show()
def predict_sentiment(review):

    # Clean review
    review = clean_text(review)

    # Tokenize
    tokens = tokenize(review)

    # Convert and pad
    sequence = pad_sequence(tokens)

    # Tensor
    tensor = torch.tensor(
        sequence,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    # Prediction
    model.eval()

    with torch.no_grad():

        output = model(tensor)

        probability = torch.sigmoid(
            output
        ).item()

    if probability >= 0.5:

        sentiment = "Positive"
        confidence = probability

    else:

        sentiment = "Negative"
        confidence = 1 - probability

    print("\n======================================")
    print("        SENTIMENT PREDICTION")
    print("======================================")

    print(
        "Review:",
        review
    )

    print(
        "Predicted Sentiment:",
        sentiment
    )

    print(
        f"Confidence: {confidence * 100:.2f}%"
    )

    print("======================================")
new_review = "The movie was excellent and very enjoyable."

predict_sentiment(new_review)

Device: cpu


HTTPError: HTTP Error 404: Not Found